# Data Preprocessing

In [ ]:
import numpy as np
import pandas as pd

def preprocess_data(train_data):
    """
    Processes training data by clamping outliers based on the IQR method.

    Args:
        train_data (np.ndarray): The training time series data (samples x timesteps).
                                 May contain existing NaNs.

    Returns:
        np.ndarray: Training data with outliers clamped to IQR bounds.
                    Original NaNs are preserved.
        float: Lower bound used for clamping.
        float: Upper bound used for clamping.
    """
    print("Starting preprocessing...")
    # Flatten the data to compute global quantiles, ignoring existing NaNs
    all_values = train_data[~np.isnan(train_data)].flatten()

    if len(all_values) == 0:
        print("Warning: Training data contains only NaNs or is empty after ignoring NaNs.")
        # Handle case with no valid data - return original data and infinite bounds
        return train_data, -np.inf, np.inf

    # Calculate Q1, Q3, and IQR
    q1 = np.percentile(all_values, 25)
    q3 = np.percentile(all_values, 75)
    iqr = q3 - q1

    # Define outlier bounds
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    print(f"Calculated bounds for outlier clamping: Lower={lower_bound}, Upper={upper_bound}")

    # Create a copy to avoid modifying the original array
    processed_data = train_data.copy()

    # Identify outliers (excluding existing NaNs)
    # This mask is mainly for reporting purposes now
    outlier_mask = ~np.isnan(processed_data) & ((processed_data < lower_bound) | (processed_data > upper_bound))
    num_outliers = np.sum(outlier_mask)
    print(f"Identified {num_outliers} outliers (excluding NaNs).")

    # Clamp outliers directly to the calculated bounds.
    # np.clip handles NaNs correctly (leaves them as NaN).
    print(f"Clamping values outside ({lower_bound}, {upper_bound}) to the bounds.")
    processed_data = np.clip(processed_data, lower_bound, upper_bound)

    if np.isnan(processed_data).any():
        print("Warning: Original NaNs detected in training data after clamping. Ensure imputation methods can handle them or add a pre-imputation step.")


    print("Preprocessing finished.")
    return processed_data, lower_bound, upper_bound


In [2]:
train_data_raw = np.array(pd.read_csv('/kaggle/input/master-1-data-challenge-ts-imputation-step-1/train.csv')) # Shape (3127, 12864)
test_data_step_a_raw = np.array(pd.read_csv('/kaggle/input/master-1-data-challenge-ts-imputation-step-1/test.csv')) # Shape (3127, 12864) with NaNs
test_data_step_b_raw = np.array(pd.read_csv('/kaggle/input/dataset-stepb/test.csv')) # Shape (1098, 12864) with NaNs


In [3]:
train_data_processed, lb, ub = preprocess_data(train_data_raw)
test_data_step_a = np.clip(test_data_step_a_raw, lb, ub)
test_data_step_b = np.clip(test_data_step_b_raw, lb, ub)

Starting preprocessing...
Calculated bounds for outlier clamping: Lower=-0.6295000000000001, Upper=1.3745
Identified 3888237 outliers (excluding NaNs).
Clamping values outside (-0.6295000000000001, 1.3745) to the bounds.
Preprocessing finished.


# Nearest Neighbor Method
Its result is used as input for our main method

In [4]:
import numpy as np
from sklearn.metrics import mean_absolute_error
from tqdm import tqdm
import multiprocessing
from joblib import Parallel, delayed

def _impute_fragment(fragment_data, fragment_indices, train_data):
    """
    Imputes missing values for a fragment of the test data.
    
    Args:
        fragment_data (np.ndarray): Fragment of test data to impute
        fragment_indices (list): Original indices of the fragment rows
        train_data (np.ndarray): Complete training data
        
    Returns:
        tuple: (fragment_indices, imputed_fragment_data)
    """
    imputed_fragment = fragment_data.copy()
    n_fragment_samples = fragment_data.shape[0]
    n_train_samples = train_data.shape[0]
    
    for i in range(n_fragment_samples):
        test_series = fragment_data[i, :]
        missing_mask = np.isnan(test_series)
        observed_mask = ~missing_mask
        
        # Skip if no missing values in this series
        if not np.any(missing_mask):
            continue
            
        # Skip if no observed values to compare
        if not np.any(observed_mask):
            continue
            
        min_mae = np.inf
        best_neighbor_idx = -1
        
        # Find the best neighbor in the training set
        for j in range(n_train_samples):
            train_series = train_data[j, :]
            
            # Calculate MAE only on observed values
            observed_indices = np.where(observed_mask)[0]
            test_observed = test_series[observed_indices]
            train_compared = train_series[observed_indices]
            
            # Check if training series has NaNs at comparison points
            if np.isnan(train_compared).any():
                valid_comparison_mask = ~np.isnan(train_compared)
                if np.sum(valid_comparison_mask) == 0:
                    continue
                mae = mean_absolute_error(test_observed[valid_comparison_mask], 
                                          train_compared[valid_comparison_mask])
            else:
                if len(test_observed) > 0:
                    mae = mean_absolute_error(test_observed, train_compared)
                else:
                    mae = np.inf
                    
            if mae < min_mae:
                min_mae = mae
                best_neighbor_idx = j
                
        # Impute missing values using best neighbor
        if best_neighbor_idx != -1:
            best_neighbor_series = train_data[best_neighbor_idx, :]
            imputed_fragment[i, missing_mask] = best_neighbor_series[missing_mask]
        else:
            # Fallback: impute with mean of observed values
            series_mean = np.nanmean(test_series)
            if np.isnan(series_mean):
                series_mean = 0
            imputed_fragment[i, missing_mask] = series_mean
            
    # Final check for remaining NaNs
    if np.isnan(imputed_fragment).any():
        fragment_mean = np.nanmean(imputed_fragment)
        if np.isnan(fragment_mean):
            fragment_mean = 0
        imputed_fragment = np.nan_to_num(imputed_fragment, nan=fragment_mean)
        
    return fragment_indices, imputed_fragment

def household_nn_imputation_parallel(train_data, test_data, n_cores=48):
    """
    Imputes missing values in test_data using the Household Nearest-Neighbor method
    with parallel processing across multiple CPU cores.

    Args:
        train_data (np.ndarray): The complete training time series data (samples x timesteps).
        test_data (np.ndarray): The test time series data with missing values (NaN).
        n_cores (int): Number of CPU cores to use for parallel processing.

    Returns:
        np.ndarray: The test data with missing values imputed.
    """
    print(f"Starting Parallelized Household Nearest-Neighbor Imputation using {n_cores} cores...")
    
    n_test_samples, n_timesteps = test_data.shape
    if train_data.shape[1] != n_timesteps:
        raise ValueError("Train and test data must have the same number of timesteps.")
        
    if np.isnan(train_data).any():
        print("Warning: Training data contains NaNs. Ensure it's fully imputed before using this method.")
    
    # Determine fragment size and create fragments
    samples_per_core = int(np.ceil(n_test_samples / n_cores))
    print(f"Total test samples: {n_test_samples}, samples per core: {samples_per_core}")
    
    # Prepare data fragments and their original indices
    fragments = []
    fragment_indices_list = []
    
    for i in range(0, n_test_samples, samples_per_core):
        end_idx = min(i + samples_per_core, n_test_samples)
        fragment = test_data[i:end_idx, :]
        fragment_indices = list(range(i, end_idx))
        
        fragments.append(fragment)
        fragment_indices_list.append(fragment_indices)
    
    actual_n_fragments = len(fragments)
    print(f"Created {actual_n_fragments} fragments for parallel processing")
    
    # Process fragments in parallel
    results = Parallel(n_jobs=n_cores, verbose=10)(
        delayed(_impute_fragment)(fragment, indices, train_data) 
        for fragment, indices in zip(fragments, fragment_indices_list)
    )
    
    # Reconstruct the imputed data in the correct order
    imputed_test_data = np.zeros_like(test_data)
    for indices, imputed_fragment in results:
        for i, original_idx in enumerate(indices):
            imputed_test_data[original_idx] = imputed_fragment[i]
    
    print("Parallelized Household Nearest-Neighbor Imputation finished.")
    
    # Final check for any remaining NaNs
    if np.isnan(imputed_test_data).any():
        print("Warning: Some NaNs remain after imputation. Filling with global mean.")
        global_mean = np.nanmean(imputed_test_data)
        if np.isnan(global_mean):
            global_mean = 0
        imputed_test_data = np.nan_to_num(imputed_test_data, nan=global_mean)
    
    return imputed_test_data



In [5]:
print("\nRunning Parallel NN Imputation for Step A...")
imputed_a_nn = household_nn_imputation_parallel(train_data_processed, test_data_step_a)
print(f"Imputed Step A shape: {imputed_a_nn.shape}, Contains NaNs: {np.isnan(imputed_a_nn).any()}")

print("\nRunning Parallel NN Imputation for Step B...")
imputed_b_nn = household_nn_imputation_parallel(train_data_processed, test_data_step_b)
print(f"Imputed Step B shape: {imputed_b_nn.shape}, Contains NaNs: {np.isnan(imputed_b_nn).any()}")

# # Save the imputed data
np.save('imputed_test_a_nn_m1.npy', imputed_a_nn)
np.save('imputed_test_b_nn_m1.npy', imputed_b_nn)


Running Parallel NN Imputation for Step A...
Starting Parallelized Household Nearest-Neighbor Imputation using 48 cores...
Total test samples: 3127, samples per core: 66
Created 48 fragments for parallel processing


[Parallel(n_jobs=48)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=48)]: Done   3 out of  48 | elapsed:  2.0min remaining: 30.1min
[Parallel(n_jobs=48)]: Done   8 out of  48 | elapsed:  2.0min remaining: 10.2min
[Parallel(n_jobs=48)]: Done  13 out of  48 | elapsed:  2.0min remaining:  5.5min
[Parallel(n_jobs=48)]: Done  18 out of  48 | elapsed:  2.1min remaining:  3.4min
[Parallel(n_jobs=48)]: Done  23 out of  48 | elapsed:  2.1min remaining:  2.2min
[Parallel(n_jobs=48)]: Done  28 out of  48 | elapsed:  2.1min remaining:  1.5min
[Parallel(n_jobs=48)]: Done  33 out of  48 | elapsed:  2.1min remaining:   56.3s
[Parallel(n_jobs=48)]: Done  38 out of  48 | elapsed:  2.1min remaining:   32.7s
[Parallel(n_jobs=48)]: Done  43 out of  48 | elapsed:  2.1min remaining:   14.4s
[Parallel(n_jobs=48)]: Done  48 out of  48 | elapsed:  2.1min finished


Parallelized Household Nearest-Neighbor Imputation finished.
Imputed Step A shape: (3127, 12864), Contains NaNs: False

Running Parallel NN Imputation for Step B...
Starting Parallelized Household Nearest-Neighbor Imputation using 48 cores...
Total test samples: 1098, samples per core: 23
Created 48 fragments for parallel processing


[Parallel(n_jobs=48)]: Using backend LokyBackend with 48 concurrent workers.
[Parallel(n_jobs=48)]: Done   3 out of  48 | elapsed:   38.9s remaining:  9.7min
[Parallel(n_jobs=48)]: Done   8 out of  48 | elapsed:   39.5s remaining:  3.3min
[Parallel(n_jobs=48)]: Done  13 out of  48 | elapsed:   39.9s remaining:  1.8min
[Parallel(n_jobs=48)]: Done  18 out of  48 | elapsed:   39.9s remaining:  1.1min
[Parallel(n_jobs=48)]: Done  23 out of  48 | elapsed:   40.0s remaining:   43.5s
[Parallel(n_jobs=48)]: Done  28 out of  48 | elapsed:   40.1s remaining:   28.6s
[Parallel(n_jobs=48)]: Done  33 out of  48 | elapsed:   40.2s remaining:   18.3s
[Parallel(n_jobs=48)]: Done  38 out of  48 | elapsed:   40.2s remaining:   10.6s
[Parallel(n_jobs=48)]: Done  43 out of  48 | elapsed:   40.3s remaining:    4.7s
[Parallel(n_jobs=48)]: Done  48 out of  48 | elapsed:   40.9s finished


Parallelized Household Nearest-Neighbor Imputation finished.
Imputed Step B shape: (1098, 12864), Contains NaNs: False


In [10]:
testa_m1 = pd.DataFrame(imputed_a_nn)
testa_m1.to_csv("testa_m1.csv")

In [11]:
testb_m1 = pd.DataFrame(imputed_b_nn)
testb_m1.to_csv("testb_m1.csv")

# GPU Accelerated STL
Our best imputation method

In [4]:
import numpy as np
import torch
import torch.nn.functional as F
from tqdm import tqdm

# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"CUDA available. Using GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("CUDA not available. Using CPU.")

def moving_average_1d(data_tensor, kernel_size):
    """ Applies a 1D moving average using convolution. Handles boundaries with padding. """
    # Ensure kernel size is odd for centering
    if kernel_size % 2 == 0:
        kernel_size += 1

    padding = kernel_size // 2
    # Create average kernel
    kernel = torch.ones(1, 1, kernel_size, device=data_tensor.device) / kernel_size
    # Reshape data for conv1d: (batch, channels, length)
    data_reshaped = data_tensor.unsqueeze(1) # Add channel dimension
    # Apply convolution
    smoothed = F.conv1d(data_reshaped, kernel, padding=padding, groups=1)
    return smoothed.squeeze(1) # Remove channel dimension

def median_seasonal_component(residuals_tensor, period):
    """ Calculates the median seasonal component for a given period. """
    batch_size, length = residuals_tensor.shape
    num_periods = (length + period - 1) // period # Calculate number of full/partial periods
    # Pad residuals to be a multiple of period length for easier reshaping
    padded_length = num_periods * period
    padding_size = padded_length - length
    # Pad at the end
    residuals_padded = F.pad(residuals_tensor, (0, padding_size), mode='constant', value=torch.nan)

    # Reshape to (batch_size, num_periods, period)
    residuals_reshaped = residuals_padded.view(batch_size, num_periods, period)

    # Calculate median across the num_periods dimension for each phase in the period
    # Handle NaNs during median calculation
    seasonal_pattern = torch.nanmedian(residuals_reshaped, dim=1).values # Shape: (batch_size, period)

    # Expand the seasonal pattern back to the original length
    # Repeat the pattern num_periods times and then truncate
    seasonal_full = seasonal_pattern.repeat(1, num_periods)[:, :length] # Shape: (batch_size, length)

    return seasonal_full


def gpu_stl_imputation(initial_imputed_data,
                         missing_mask_original,
                         periods=[48, 48*7], # Daily and Weekly
                         kernel_sizes=[101], # Example kernel size for trend
                         iterations=30):
    """
    Performs iterative STL-like imputation using PyTorch for potential GPU acceleration.

    Args:
        initial_imputed_data (np.ndarray): Test data with initial imputation
                                           (e.g., from Method 2 or simple mean/median).
                                           Should have no NaNs. (samples x timesteps).
        missing_mask_original (np.ndarray): Boolean mask indicating originally missing
                                            positions (True where missing). (samples x timesteps).
        periods (list[int]): List of seasonal periods to cycle through.
        kernel_sizes (list[int]): List of kernel sizes for moving average (trend)
                                  to cycle through.
        iterations (int): Number of refinement iterations.

    Returns:
        np.ndarray: The final imputed data after iterations.
    """
    print("Starting GPU-Accelerated STL Imputation...")
    if np.isnan(initial_imputed_data).any():
        raise ValueError("Initial imputed data must not contain NaNs.")
    if initial_imputed_data.shape != missing_mask_original.shape:
        raise ValueError("Data shape and mask shape must match.")

    # Convert data and mask to PyTorch tensors and move to device
    X = torch.tensor(initial_imputed_data, dtype=torch.float32).to(device)
    M_original_inv = torch.tensor(~missing_mask_original, dtype=torch.float32).to(device) # 1 where observed
    M_original = torch.tensor(missing_mask_original, dtype=torch.float32).to(device)     # 1 where missing

    num_periods = len(periods)
    num_kernels = len(kernel_sizes)

    for i in tqdm(range(iterations), desc="STL Iterations"):
        p = periods[i % num_periods]
        k = kernel_sizes[i % num_kernels]

        # 1. Trend Extraction (Moving Average)
        T = moving_average_1d(X, kernel_size=k)

        # 2. Seasonal Estimation
        R = X - T # Residuals (or Detrended series)
        Sp_full = median_seasonal_component(R, period=p) # Seasonal component based on medians

        # 3. Reconstruction
        Y = T + Sp_full

        # 4. Update only originally missing values
        # X_new = Y where mask is 1 (missing), X_old where mask is 0 (observed)
        X = (M_original * Y) + (M_original_inv * X)

        # Optional: Add a check for NaNs/Infs introduced during computation
        if torch.isnan(X).any() or torch.isinf(X).any():
            print(f"Warning: NaNs or Infs detected during iteration {i+1}. Clamping.")
            X = torch.nan_to_num(X, nan=0.0, posinf=1e6, neginf=-1e6) # Simple clamp/fill


    print("GPU-Accelerated STL Imputation finished.")
    # Convert back to NumPy array on CPU
    final_imputed_data = X.cpu().numpy()
    return final_imputed_data


CUDA available. Using GPU: NVIDIA L4


In [ ]:
print("\nRunning STL Imputation for Step A...")
# Use the output of NN as initial imputation
initial_imputation_a = np.array(pd.read_csv("/kaggle/input/m1-imputation/testa_m1.csv").drop(columns=['Unnamed: 0']))
mask_a_original = np.isnan(test_data_step_a_raw) # Use the raw mask before clamping/NN
imputed_a_stl = gpu_stl_imputation(initial_imputation_a, mask_a_original, iterations=100) 
print(f"Imputed Step A shape: {imputed_a_stl.shape}, Contains NaNs: {np.isnan(imputed_a_stl).any()}")

print("\nRunning STL Imputation for Step B...")
# Use the output of NN as initial imputation
initial_imputation_b = np.array(pd.read_csv("/kaggle/input/m1-imputation/testb_m1.csv").drop(columns=['Unnamed: 0']))
mask_b_original = np.isnan(test_data_step_b_raw) # Use the raw mask before clamping/NN
imputed_b_stl = gpu_stl_imputation(initial_imputation_b, mask_b_original, iterations=100) 
print(f"Imputed Step B shape: {imputed_b_stl.shape}, Contains NaNs: {np.isnan(imputed_b_stl).any()}")

pd.DataFrame(imputed_a_stl).to_csv("testa_m2.csv")
pd.DataFrame(imputed_b_stl).to_csv("testb_m2.csv")


Running STL Imputation for Step A...
Starting GPU-Accelerated STL Imputation...


STL Iterations: 100%|██████████| 100/100 [00:03<00:00, 25.63it/s]


GPU-Accelerated STL Imputation finished.
Imputed Step A shape: (3127, 12864), Contains NaNs: False

Running STL Imputation for Step B...
Starting GPU-Accelerated STL Imputation...


STL Iterations: 100%|██████████| 100/100 [00:01<00:00, 73.35it/s]


GPU-Accelerated STL Imputation finished.
Imputed Step B shape: (1098, 12864), Contains NaNs: False
